# Affinity Classification Diagnostics

This notebook computes ordinal affinity diagnostics on `(abstract, question)` pairs.

Metrics included:
1. Per-model class metrics
2. Per-model quadratic weighted kappa
3. Per-model mean signed error
4. An additional synthetic model formed by averaging predictions across models

In [10]:
from pathlib import Path

import importlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [11]:
# Label definitions (edit only if your taxonomy changes).
LABEL_ORDER = ["Low", "Moderate", "High"]
LABEL_TO_NUM = {"Low": 0, "Moderate": 1, "High": 2}
NUM_TO_LABEL = {0: "Low", 1: "Moderate", 2: "High"}


def normalize_label(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, np.integer)):
        return NUM_TO_LABEL.get(int(v), np.nan)
    text = str(v).strip()
    text_low = text.lower()
    if text_low in {"low", "moderate", "high"}:
        return text_low.capitalize()
    return np.nan


# Hardcoded column names used throughout this notebook.
expert_col = "Evaluator_Affinity_Level"
prediction_col = "affinity_level"
abstract_id_col = "Abstract_Index"
model_col = "Model_Slug"

## Load Data

The loader tries likely project files first (including benchmark and calibration outputs), then you can override `input_path` manually if needed.

In [12]:
repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
bench_root = repo_root / "data" / "results" / "affinity_benchmark"
calib_root = repo_root / "data" / "results" / "affinity_calibration"
benchmark_input_filename = "adjudicated_ra_model_rows.csv"
calibration_input_filename = "calibration_abstract_question_affinities.csv"

# benchmark_run_dir_names = ["20260422_212121_1", "20260422_212121_4", "20260422_212121_3"]
benchmark_run_dir_names = ["20260726_134607_1"]
# benchmark_run_dir_names = ["20260726_154151_1"]
    
calibration_run_dir_names = ["20260728_abstracts_evaluation_template_mark_om_v1"]

def load_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin1"):
            try:
                return pd.read_csv(path, encoding=encoding)
            except UnicodeDecodeError:
                continue
        raise UnicodeDecodeError(
            "utf-8", b"", 0, 1, f"Unable to decode CSV file: {path}"
        )
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file extension for {path}")

def collect_frames(root: Path, run_dir_names, input_filename: str, source_name: str):
    if isinstance(run_dir_names, (str, Path)):
        run_dir_names = [run_dir_names]

    run_dirs = sorted([p for p in root.iterdir() if p.is_dir()])
    if not run_dirs:
        raise FileNotFoundError(f"No run directories found under {root}")

    selected_run_dirs = [p for p in run_dirs if p.name in {str(x) for x in run_dir_names}]
    if not selected_run_dirs:
        raise FileNotFoundError(f"No matching run directories found for: {run_dir_names}")

    frames = []
    for run_dir in selected_run_dirs:
        input_path = run_dir / input_filename
        if not input_path.exists():
            raise FileNotFoundError(f"Missing file: {input_path}")

        run_df = load_table(input_path)
        run_df["Run_ID"] = run_dir.name
        run_df["Source_Type"] = source_name
        frames.append(run_df)

    return frames, selected_run_dirs

benchmark_frames, selected_benchmark_dirs = collect_frames(
    bench_root, benchmark_run_dir_names, benchmark_input_filename, "benchmark"
)
calibration_frames, selected_calibration_dirs = collect_frames(
    calib_root, calibration_run_dir_names, calibration_input_filename, "calibration"
)

bench_df = pd.concat(benchmark_frames, ignore_index=True)
calib_df = pd.concat(calibration_frames, ignore_index=True)
df_raw = pd.merge(
    bench_df,
    calib_df,
    on=["Abstract_Index", "RA2025_ID"],
    how="inner",
    suffixes=("_bench", "_calib"),
)

print("Benchmark run directories:", [p.name for p in selected_benchmark_dirs])
print("Calibration run directories:", [p.name for p in selected_calibration_dirs])
print("Calibration abstracts:", calib_df['Abstract_Index'].nunique())
print("Benchmark abstracts:", bench_df['Abstract_Index'].nunique())
print("Rows loaded:", len(df_raw))
print(f"Columns ({len(df_raw.columns)}): {list(df_raw.columns)}")
display(df_raw.head(10))

Benchmark run directories: ['20260726_134607_1']
Calibration run directories: ['20260728_abstracts_evaluation_template_mark_om_v1']
Calibration abstracts: 91
Benchmark abstracts: 105
Rows loaded: 3003
Columns (23): ['Abstract_Index', 'Document Title_bench', 'RA2025_ID', 'RA_Question_bench', 'Cosine_x100', 'LLM_Affinity', 'LLM_Affinity_Reason', 'Model_Slug', 'Run_ID_bench', 'Abstract_Index_Norm', 'affinity_level', 'Final_Affinity', 'Final_Method', 'Final_Reason', 'Source_Type_bench', 'Document Title_calib', 'RA_Question_calib', 'Evaluator_Affinity', 'Evaluator_Affinity_Level', 'Evaluator_Affinity_Reason', 'Source_Evaluator', 'Run_ID_calib', 'Source_Type_calib']


,Abstract_Index,Document Title_bench,RA2025_ID,RA_Question_bench,Cosine_x100,LLM_Affinity,LLM_Affinity_Reason,Model_Slug,Run_ID_bench,Abstract_Index_Norm,...,Final_Reason,Source_Type_bench,Document Title_calib,RA_Question_calib,Evaluator_Affinity,Evaluator_Affinity_Level,Evaluator_Affinity_Reason,Source_Evaluator,Run_ID_calib,Source_Type_calib
0,10.1109/TPWRS.2023.3292799,Optimal Primary Frequency Reserve Provision by...,35,"what models and methods, including testing, ar...",49.4778,70.0,Addresses ability of DERs (including variable ...,deepseek-v4-flash-cloud,20260726_134607_1,10.1109/TPWRS.2023.3292799,...,Signals ranged from low to moderately relevant...,benchmark,Optimal Primary Frequency Reserve Provision by...,"What models and methods, including testing, ar...",70,Moderate,NaN,Abstracts Evaluation Template - Mark OM.xlsm,20260728_abstracts_evaluation_template_mark_om_v1,calibration
1,10.1109/TPWRS.2023.3292799,Optimal Primary Frequency Reserve Provision by...,18,what are the relative merits of different coor...,40.4109,80.0,Directly addresses coordination architectures ...,deepseek-v4-flash-cloud,20260726_134607_1,10.1109/TPWRS.2023.3292799,...,Most models saw strong relevance because the p...,benchmark,Optimal Primary Frequency Reserve Provision by...,What are the relative merits of different coor...,10,Low,NaN,Abstracts Evaluation Template - Mark OM.xlsm,20260728_abstracts_evaluation_template_mark_om_v1,calibration
2,10.1109/TPWRS.2023.3292799,Optimal Primary Frequency Reserve Provision by...,12,what mechanisms and appropriate aggregate der ...,39.5109,85.0,Focuses on aggregate DER representations and d...,deepseek-v4-flash-cloud,20260726_134607_1,10.1109/TPWRS.2023.3292799,...,There is strong consensus that this is directl...,benchmark,Optimal Primary Frequency Reserve Provision by...,What mechanisms and appropriate aggregate DER ...,50,Moderate,NaN,Abstracts Evaluation Template - Mark OM.xlsm,20260728_abstracts_evaluation_template_mark_om_v1,calibration
3,10.1109/TPWRS.2023.3292799,Optimal Primary Frequency Reserve Provision by...,15,"what aggregate quantities must be monitored, s...",36.7361,80.0,Addresses monitoring and validation of aggrega...,deepseek-v4-flash-cloud,20260726_134607_1,10.1109/TPWRS.2023.3292799,...,Most models identified strong relevance becaus...,benchmark,Optimal Primary Frequency Reserve Provision by...,"What aggregate quantities must be monitored, s...",10,Low,NaN,Abstracts Evaluation Template - Mark OM.xlsm,20260728_abstracts_evaluation_template_mark_om_v1,calibration
4,10.1109/TPWRS.2023.3292799,Optimal Primary Frequency Reserve Provision by...,17,what is a scalable data architecture for der m...,20.6269,50.0,Scalable data architecture for DER monitoring ...,deepseek-v4-flash-cloud,20260726_134607_1,10.1109/TPWRS.2023.3292799,...,All models agreed this is only weakly related....,benchmark,Optimal Primary Frequency Reserve Provision by...,What is a scalable data architecture for DER m...,10,Low,NaN,Abstracts Evaluation Template - Mark OM.xlsm,20260728_abstracts_evaluation_template_mark_om_v1,calibration
5,10.1109/TPWRS.2023.3292799,Optimal Primary Frequency Reserve Provision by...,16,what is the communication capability needed to...,18.2876,30.0,Communication capability for DER coordination ...,deepseek-v4-flash-cloud,20260726_134607_1,10.1109/TPWRS.2023.3292799,...,NaN,benchmark,Optimal Primary Frequency Reserve Provision by...,What is the communication capability needed to...,10,Low,NaN,Abstracts Evaluation Template - Mark OM.xlsm,20260728_abstracts_evaluation_template_mark_om_v1,calibration
6,10.1109/TPWRS.2023.3242715,Graph Learning-Based Voltage Regulation in Dis...,13,how can system operators quantify the bulk-lev...,41.3375,30.0,Abstract addresses voltage regulation service ...,deepseek-v4-flash-cloud,20260726_134607_1,10.1109/TPWRS.2023.3242715,...,Most models agreed the overlap is limited to D...,benchmark,Graph Learning-Based Voltage Regulation in Dis...,How can system operators qua

In [13]:
required_cols = [expert_col, prediction_col, abstract_id_col]
missing_required = [c for c in required_cols if c not in df_raw.columns]

print("Hardcoded column mapping in use:")
print(f"  expert_col      -> {expert_col}")
print(f"  prediction_col  -> {prediction_col}")
print(f"  abstract_id_col -> {abstract_id_col}")

if missing_required:
    raise ValueError(
        f"Missing required columns in df_raw: {missing_required}. "
        "Please update hardcoded column names near the top of this notebook."
    )

Hardcoded column mapping in use:
  expert_col      -> Evaluator_Affinity_Level
  prediction_col  -> affinity_level
  abstract_id_col -> Abstract_Index


## Clean Labels

The cleaner normalizes variants such as `low`, `LOW`, `med`, `medium`, and `high`, then maps labels to ordinal values:

- Low = 0
- Medium = 1
- High = 2

In [14]:
# Assume labels are canonical: 'Low', 'Moderate', 'High'.
eval_df = df_raw.copy()
rows_before = len(eval_df)

expert_labels = eval_df[expert_col].map(normalize_label)
pred_labels = eval_df[prediction_col].map(normalize_label)
valid_mask = expert_labels.notna() & pred_labels.notna()

eval_df = eval_df.loc[valid_mask].copy()
eval_df['expert_num'] = expert_labels.loc[valid_mask].map(LABEL_TO_NUM).astype(int).to_numpy()
eval_df['pred_num'] = pred_labels.loc[valid_mask].map(LABEL_TO_NUM).astype(int).to_numpy()

rows_after = len(eval_df)
rows_removed = rows_before - rows_after

print(f'Rows before cleaning: {rows_before:,}')
print(f'Rows after cleaning:  {rows_after:,}')
print(f'Rows removed:         {rows_removed:,}')

print(f'Evaluated (abstract, question) pairs after cleaning: {rows_after:,}')
if abstract_id_col in eval_df.columns:
    print(f'Unique abstracts: {eval_df[abstract_id_col].nunique():,}')
else:
    print('Unique abstracts: not available (abstract_id_col not found).')
print(f'Unique abstract indices: {df_raw.Abstract_Index.unique().size:,}')

Rows before cleaning: 3,003
Rows after cleaning:  3,003
Rows removed:         0
Evaluated (abstract, question) pairs after cleaning: 3,003
Unique abstracts: 91
Unique abstract indices: 91


## Model-wise Metrics

In [15]:
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

affinity_metrics = importlib.import_module("src.analysis.affinity_metrics")
AVERAGE_MODEL_NAME = "Average across models"


def build_average_model_frame(df: pd.DataFrame) -> pd.DataFrame:
    averaged_df = (
        df.groupby([abstract_id_col, "RA2025_ID"], as_index=False)
        .agg(
            expert_num=("expert_num", "first"),
            pred_num=("pred_num", "mean"),
        )
    )
    averaged_df["pred_num"] = np.floor(averaged_df["pred_num"] + 0.5).clip(0, 2).astype(int)
    return averaged_df


def build_metrics_for_model(df: pd.DataFrame, model_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    y_true = df["expert_num"].to_numpy()
    y_pred = df["pred_num"].to_numpy()

    class_metrics_df = affinity_metrics.build_affinity_class_metrics(
        y_true,
        y_pred,
        labels=list(LABEL_TO_NUM.values()),
        class_names=LABEL_ORDER,
    ).copy()
    class_metrics_df.insert(0, "Model", model_name)

    overall_metrics_df = pd.DataFrame(
        [
            {
                "Model": model_name,
                "Quadratic Weighted Kappa": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
                "Mean signed error": float(np.mean(y_pred - y_true)),
                "Pairs": int(len(df)),
                "Unique abstracts": int(df[abstract_id_col].nunique()),
            }
        ]
    )
    return class_metrics_df, overall_metrics_df

In [16]:
# Compare each model against the "Average across models" baseline, per class.
# Support is excluded here: it's a sample count, not a higher-is-better metric,
# so "better/worse than average" doesn't apply to it.
delta_metrics = ["Precision", "Recall", "One_vs_Rest_Accuracy"]
colors = {"Low": "#636EFA", "Moderate": "#EF553B", "High": "#00CC96"}

average_metrics_df = (
    class_metrics_df.loc[class_metrics_df["Model"] == AVERAGE_MODEL_NAME, ["Class"] + delta_metrics]
    .rename(columns={metric: f"{metric}_avg" for metric in delta_metrics})
)

delta_df = class_metrics_df[class_metrics_df["Model"] != AVERAGE_MODEL_NAME].merge(
    average_metrics_df, on="Class", how="left"
)
for metric in delta_metrics:
    delta_df[f"{metric}_delta"] = delta_df[metric] - delta_df[f"{metric}_avg"]

model_order = sorted(delta_df["Model"].unique())

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f"{m} (Δ vs average)" for m in delta_metrics],
    horizontal_spacing=0.08,
)

for col_idx, metric in enumerate(delta_metrics, start=1):
    for class_label in LABEL_ORDER:
        class_data = (
            delta_df[delta_df["Class"] == class_label]
            .set_index("Model")
            .reindex(model_order)
            .reset_index()
        )
        fig.add_trace(
            go.Bar(
                x=class_data["Model"],
                y=class_data[f"{metric}_delta"],
                name=class_label,
                marker=dict(color=colors[class_label], line=dict(color="#1f1f1f", width=0.5)),
                legendgroup=class_label,
                showlegend=(col_idx == 1),
                customdata=np.stack([class_data[metric], class_data[f"{metric}_avg"]], axis=-1),
                hovertemplate=(
                    f"<b>{class_label}</b><br>"
                    "Model: %{x}<br>"
                    f"{metric}: %{{customdata[0]:.3f}}<br>"
                    "Average: %{customdata[1]:.3f}<br>"
                    "Δ: %{y:+.3f}"
                    "<extra></extra>"
                ),
            ),
            row=1,
            col=col_idx,
        )
    fig.add_hline(
        y=0,
        line_width=1,
        line_color="#1f1f1f",
        annotation_text="Average",
        annotation_position="top left",
        row=1,
        col=col_idx,
    )
    fig.update_yaxes(title_text="Δ vs average", row=1, col=col_idx)
    fig.update_xaxes(title_text="Model", row=1, col=col_idx)

fig.update_layout(
    title_text="Model Performance Relative to Cross-Model Average",
    barmode="group",
    bargap=0.2,
    bargroupgap=0.05,
    height=500,
    hovermode="closest",
)

fig.show()


In [17]:
model_class_metrics_frames = []
model_overall_metrics_frames = []

for model_name, model_df in eval_df.groupby(model_col, sort=True):
    class_metrics_df, overall_metrics_df = build_metrics_for_model(model_df, model_name)
    model_class_metrics_frames.append(class_metrics_df)
    model_overall_metrics_frames.append(overall_metrics_df)

average_model_df = build_average_model_frame(eval_df)
average_class_metrics_df, average_overall_metrics_df = build_metrics_for_model(
    average_model_df,
    AVERAGE_MODEL_NAME,
)

class_metrics_df = pd.concat(
    model_class_metrics_frames + [average_class_metrics_df],
    ignore_index=True,
)
overall_metrics_df = pd.concat(
    model_overall_metrics_frames + [average_overall_metrics_df],
    ignore_index=True,
)

class_metrics_df["Class"] = pd.Categorical(class_metrics_df["Class"], categories=LABEL_ORDER, ordered=True)
class_metrics_df = class_metrics_df.sort_values(["Model", "Class"]).reset_index(drop=True)
overall_metrics_df = overall_metrics_df.sort_values("Model").reset_index(drop=True)

display(overall_metrics_df)
display(class_metrics_df[["Model", "Class", "Support", "Precision", "Recall", "One_vs_Rest_Accuracy"]])

,Model,Quadratic Weighted Kappa,Mean signed error,Pairs,Unique abstracts
0,Average across models,0.413280,0.345515,602,91
1,deepseek-v4-flash-cloud,0.402440,0.164452,602,91
2,gemini-3.1-flash-lite,0.323367,0.533223,602,91
3,gemma4-31b-cloud,0.357379,0.400332,602,91
4,gpt-5.4-mini,0.354073,0.151163,602,91
5,mistral-large-3-675b-cloud,0.361165,0.410084,595,90


,Model,Class,Support,Precision,Recall,One_vs_Rest_Accuracy
0,Average across models,Low,438,0.900709,0.579909,0.647841
1,Average across models,Moderate,137,0.282158,0.496350,0.598007
2,Average across models,High,27,0.227848,0.666667,0.883721
3,deepseek-v4-flash-cloud,Low,438,0.844737,0.732877,0.707641
4,deepseek-v4-flash-cloud,Moderate,137,0.350649,0.394161,0.696013
5,deepseek-v4-flash-cloud,High,27,0.250000,0.629630,0.898671
6,gemini-3.1-flash-lite,Low,438,0.930851,0.399543,0.541528
7,gemini-3.1-flash-lite,Moderate,137,0.234177,0.540146,0.493355
8,gemini-3.1-flash-lite,High,27,0.183673,0.666667,0.852159
9,gemma4-31b-cloud,Low,438,0.912351,0.522831,0.616279


# Model Recommendation — Affinity Diagnostics

## Recommendation

Keep using the **multi-model ensemble average** as the primary output rather than switching to any single model — it's the only "model" that beats every individual model on agreement with experts (QWK **0.413** vs. deepseek's **0.402**, the best individual).

If a single model is ever needed (cost/latency reasons), use **`deepseek-v4-flash-cloud`** — it's the only individual model that beats the average on both Recall and One-vs-Rest Accuracy for the Low class, and on Moderate-class accuracy, while carrying a much smaller bias than the other recall-strong models.

## Supporting Evidence

### Overall metrics

| Model | QWK | Mean Signed Error |
|---|---|---|
| **Average across models** | **0.413** | +0.346 |
| deepseek-v4-flash-cloud | 0.402 | +0.164 |
| mistral-large-3-675b-cloud | 0.361 | +0.410 |
| gemma4-31b-cloud | 0.357 | +0.400 |
| gpt-5.4-mini | 0.354 | +0.151 |
| gemini-3.1-flash-lite | 0.323 | +0.533 |

### Notable deltas vs. the ensemble average (Δ = model − average)

| Model | Class | Metric | Δ vs average |
|---|---|---|---|
| deepseek-v4-flash-cloud | Low | Recall | **+0.153** |
| deepseek-v4-flash-cloud | Low | Accuracy | **+0.060** |
| deepseek-v4-flash-cloud | Moderate | Accuracy | **+0.098** |
| gpt-5.4-mini | High | Recall | **−0.333** |
| gpt-5.4-mini | Moderate | Recall | **−0.190** |
| gemini-3.1-flash-lite | Low | Recall | **−0.180** |
| gemini-3.1-flash-lite | Low | Accuracy | **−0.106** |

## Key Findings

1. **Systemic over-prediction.** Every model has a positive mean signed error (predicts higher affinity than the expert), ranging from +0.15 (gpt-5.4-mini, deepseek) to +0.53 (gemini). Even the ensemble average sits at +0.35 — worth recalibrating the Low/Moderate/High cut points rather than treating them as fixed.
2. **gpt-5.4-mini is a trap.** Its low bias (+0.15) looks attractive, but High-class Recall is only 0.333 (Δ **−0.333** vs. average) and Moderate Recall is 0.307 (Δ **−0.190**) — it systematically misses the abstracts you'd most want to catch. The low bias comes from under-calling the top classes, not from good calibration.
3. **gemini-3.1-flash-lite and gemma4-31b-cloud are the weakest overall** — lowest two QWKs (0.323, 0.357) and consistently negative deltas vs. average on Low-class Recall/Accuracy and High-class Precision. Candidates to deprioritize or debug rather than keep in the benchmark rotation.

## Caveat

The **High class has only 27 pairs**, so its recall swings (e.g., 0.33 vs 0.67) are noisy — don't over-index on single-model High-class deltas without more calibration data.

## Suggested Next Step

Since the ensemble mean already outperforms any individual model, the highest-value follow-up is likely a **bias-corrected or weighted ensemble** (down-weighting gemini/gemma, which drag the average's bias up) rather than searching for a better single model.
